In [ ]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from prophet import Prophet
from prophet.utilities import regressor_coefficients

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# CONFIG
DATA_PATH = "/content/Dengue_Climate_Bangladesh.csv"
OUT_TABLES = "/content/"
OUT_FIGS = "/content/"

CLIMATE_COLS = ["MIN", "MAX", "HUMIDITY", "RAINFALL"]
LAG_COLS = ["y_lag12", "y_roll12"]

USE_CLIMATE_REGRESSORS = False
REGRESSORS = LAG_COLS + (CLIMATE_COLS if USE_CLIMATE_REGRESSORS else [])

TRAIN_END = "2024-12-01"
TEST_START = "2025-01-01"
TEST_END = "2025-10-01"

# Nested CV grid and fold cutoffs (train-only; see docstring point 4)
CPS_GRID = [0.05, 0.1, 0.3, 0.5, 0.7, 0.9]
RANGE_GRID = [0.8, 0.9, 1.0]
CV_FOLD_CUTOFF_YEARS = list(range(2017, 2024))
CV_HORIZON_MONTHS = 10

INTERVAL_WIDTH = 0.90

np.random.seed(42)


# 1. DATA LOADING & FEATURE ENGINEERING
def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["ds"] = pd.to_datetime(dict(year=df.YEAR, month=df.MONTH, day=1))
    df = df.sort_values("ds").reset_index(drop=True)
    df["y"] = np.log1p(df["DENGUE"])
    df["y_lag12"] = df["y"].shift(12)
    df["y_roll12"] = df["y"].shift(1).rolling(12).mean()
    df = df.dropna(subset=LAG_COLS).reset_index(drop=True)
    return df


def inv(y_log):
    """Inverse of log1p, clipped at 0 (cases cannot be negative)."""
    return np.clip(np.expm1(np.asarray(y_log)), 0, None)


# 2. METRICS
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100)
    smape = float(np.mean(2 * np.abs(y_true - y_pred) /
                           (np.abs(y_true) + np.abs(y_pred) + 1e-9)) * 100)
    r2 = r2_score(y_true, y_pred)
    yt_log, yp_log = np.log1p(y_true), np.log1p(np.clip(y_pred, 0, None))
    return dict(
        MAE=mae, RMSE=rmse, MAPE=mape, sMAPE=smape, R2=r2,
        MAE_log=mean_absolute_error(yt_log, yp_log),
        RMSE_log=np.sqrt(mean_squared_error(yt_log, yp_log)),
        R2_log=r2_score(yt_log, yp_log),
    )


# 3. MODEL
def build_prophet(changepoint_prior_scale, changepoint_range,
                   regressors=REGRESSORS):
    m = Prophet(
        growth="linear",
        yearly_seasonality=False,     # see docstring point 3
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
        changepoint_prior_scale=changepoint_prior_scale,
        changepoint_range=changepoint_range,
        interval_width=INTERVAL_WIDTH,
    )
    for r in regressors:
        m.add_regressor(r, standardize=True)
    return m


def fit_predict(train, test, changepoint_prior_scale, changepoint_range,
                 regressors=REGRESSORS, return_model=False):
    m = build_prophet(changepoint_prior_scale, changepoint_range, regressors)
    m.fit(train[["ds", "y"] + regressors])
    fc = m.predict(test[["ds"] + regressors])
    pred = inv(fc["yhat"].values)
    lower = inv(fc["yhat_lower"].values)
    upper = inv(fc["yhat_upper"].values)
    if return_model:
        return pred, lower, upper, m, fc
    return pred, lower, upper


# 4. NESTED, WALK-FORWARD CROSS-VALIDATION (hyperparameter selection)
def make_cv_folds(df, cutoff_years, horizon_months=CV_HORIZON_MONTHS):
    """
    Expanding-window folds strictly inside the training period. Each fold
    mirrors the deployment task: given everything through December of year Y,
    forecast Jan-Oct of year Y+1. Never touches the 2025 hold-out.
    """
    folds = []
    for y in cutoff_years:
        train = df[df["ds"] <= f"{y}-12-01"].copy()
        test = df[df["ds"] >= f"{y+1}-01-01"].copy().head(horizon_months)
        if len(test) == horizon_months:
            folds.append({"cutoff_year": y, "train": train, "test": test})
    return folds


def tune_hyperparameters(df):
    folds = make_cv_folds(df, CV_FOLD_CUTOFF_YEARS)
    print(f"Nested CV: {len(folds)} walk-forward folds "
          f"(train-only, cutoffs {CV_FOLD_CUTOFF_YEARS[0]}-{CV_FOLD_CUTOFF_YEARS[-1]})")

    rows = []
    for cps, crange in itertools.product(CPS_GRID, RANGE_GRID):
        fold_rmse_log = []
        for f in folds:
            pred, _, _ = fit_predict(f["train"], f["test"], cps, crange)
            m = compute_metrics(f["test"]["DENGUE"].values, pred)
            fold_rmse_log.append(m["RMSE_log"])
        rows.append(dict(changepoint_prior_scale=cps, changepoint_range=crange,
                          mean_cv_rmse_log=np.mean(fold_rmse_log),
                          std_cv_rmse_log=np.std(fold_rmse_log)))
        print(f"  cps={cps:>4} range={crange}  "
              f"mean_CV_RMSE_log={np.mean(fold_rmse_log):.3f} +/- {np.std(fold_rmse_log):.3f}")

    tuning_df = pd.DataFrame(rows).sort_values("mean_cv_rmse_log").reset_index(drop=True)
    tuning_df.to_csv(f"{OUT_TABLES}/table_S1_hyperparameter_tuning.csv", index=False)

    best = tuning_df.iloc[0]
    print(f"\n>>> Selected (nested CV, train-only): "
          f"changepoint_prior_scale={best.changepoint_prior_scale}, "
          f"changepoint_range={best.changepoint_range}")
    return float(best.changepoint_prior_scale), float(best.changepoint_range), tuning_df


# 5. FINAL, SINGLE-TOUCH HOLD-OUT EVALUATION
def run_final_holdout(df, best_cps, best_range):
    train = df[df["ds"] <= TRAIN_END].copy()
    test = df[(df["ds"] >= TEST_START) & (df["ds"] <= TEST_END)].copy()
    print(f"\nFinal fit: train={len(train)} months, test={len(test)} months "
          f"({test.ds.min().date()} to {test.ds.max().date()})")

    pred, lower, upper, model, fc_test = fit_predict(
        train, test, best_cps, best_range, return_model=True)
    fc_train = model.predict(train[["ds"] + REGRESSORS])

    y_true = test["DENGUE"].values
    metrics = compute_metrics(y_true, pred)

    with open(f"{OUT_TABLES}/final_metrics.json", "w") as f:
        json.dump({k: float(v) for k, v in metrics.items()}, f, indent=2)

    print("\n=== Final 2025 Jan-Oct hold-out metrics (single evaluation) ===")
    for k, v in metrics.items():
        print(f"  {k:10s}: {v:,.3f}")

    # generic predictions table for merging with other models in the paper
    out = pd.DataFrame({
        "ds": test["ds"].values,
        "actual": y_true,
        "Prophet_pred": pred,
        "Prophet_lower90": lower,
        "Prophet_upper90": upper,
    })
    out.to_csv(f"{OUT_TABLES}/prophet_predictions.csv", index=False)

    coefs = regressor_coefficients(model)
    coefs.to_csv(f"{OUT_TABLES}/regressor_coefficients.csv", index=False)
    print("\nRegressor coefficients:")
    print(coefs.round(4).to_string(index=False))

    return dict(train=train, test=test, pred=pred, lower=lower, upper=upper,
                model=model, fc_train=fc_train, fc_test=fc_test,
                metrics=metrics, coefs=coefs)


# 6. FIGURES
def make_figures(df, result):
    train, test = result["train"], result["test"]
    fc_train = result["fc_train"]

    train_pred = inv(fc_train["yhat"].values)
    train_lower = inv(fc_train["yhat_lower"].values)
    train_upper = inv(fc_train["yhat_upper"].values)

    full_ds = pd.concat([train["ds"], test["ds"]])
    full_actual = pd.concat([train["DENGUE"], test["DENGUE"]])
    full_pred = np.concatenate([train_pred, result["pred"]])
    full_lower = np.concatenate([train_lower, result["lower"]])
    full_upper = np.concatenate([train_upper, result["upper"]])

    # Fig 1: full timeline
    fig, ax = plt.subplots(figsize=(13, 5.5))
    ax.plot(full_ds, full_actual, color="#1f2937", lw=1.3, label="Actual dengue cases")
    ax.plot(full_ds, full_pred, color="#e11d48", lw=1.2, ls="--", label="Prophet fitted/forecast")
    ax.fill_between(full_ds, full_lower, full_upper, color="#e11d48", alpha=0.15, label=f"{int(INTERVAL_WIDTH*100)}% interval")
    ax.axvline(pd.Timestamp(TEST_START), color="#2563eb", ls=":", lw=1.5)
    ax.text(pd.Timestamp(TEST_START), ax.get_ylim()[1]*0.92, " Train/Test split", color="#2563eb", fontsize=9)
    ax.set_title("Prophet: Actual vs Fitted/Forecast Dengue Cases (2009-2025)", fontweight="bold")
    ax.set_ylabel("Monthly dengue cases"); ax.set_xlabel("Year")
    ax.legend(loc="upper left")
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.tight_layout(); fig.savefig(f"{OUT_FIGS}/fig1_full_timeline.png"); plt.close(fig)

    # Fig 2: zoomed hold-out
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    ax.plot(test["ds"], test["DENGUE"], "o-", color="#1f2937", lw=2, ms=6, label="Actual")
    ax.plot(test["ds"], result["pred"], "s--", color="#e11d48", lw=2, ms=6, label="Prophet")
    ax.fill_between(test["ds"], result["lower"], result["upper"], color="#e11d48", alpha=0.18, label=f"{int(INTERVAL_WIDTH*100)}% interval")
    ax.set_title("Final Hold-out: 2025 Jan-Oct, Actual vs Prophet", fontweight="bold")
    ax.set_ylabel("Monthly dengue cases"); ax.set_xlabel("Month (2025)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.legend()
    fig.tight_layout(); fig.savefig(f"{OUT_FIGS}/fig2_holdout_zoom.png"); plt.close(fig)

    # Fig 3: scatter (log-log)
    y_true, y_pred = test["DENGUE"].values, result["pred"]
    fig, ax = plt.subplots(figsize=(6, 6))
    mx = max(y_true.max(), y_pred.max()) * 1.2
    ax.plot([1, mx], [1, mx], color="gray", ls="--", lw=1, label="Perfect prediction")
    ax.scatter(np.clip(y_true, 1, None), np.clip(y_pred, 1, None), c=range(len(y_true)),
               cmap="viridis", s=90, edgecolor="k", zorder=3)
    for i, mth in enumerate(test["ds"].dt.strftime("%b")):
        ax.annotate(mth, (np.clip(y_true[i], 1, None) * 1.05, y_pred[i]), fontsize=8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Actual cases (log scale)"); ax.set_ylabel("Predicted cases (log scale)")
    ax.set_title(f"Actual vs Predicted (2025 hold-out)\nR2 (log scale) = {result['metrics']['R2_log']:.3f}", fontweight="bold")
    ax.legend()
    fig.tight_layout(); fig.savefig(f"{OUT_FIGS}/fig3_scatter.png"); plt.close(fig)

    # Fig 4: residuals
    resid = y_true - y_pred
    ape = np.abs(resid) / np.clip(y_true, 1, None) * 100
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(test["ds"].dt.strftime("%b"), resid, color=np.where(resid >= 0, "#16a34a", "#dc2626"))
    axes[0].axhline(0, color="black", lw=1)
    axes[0].set_title("Residuals (Actual - Predicted)", fontweight="bold"); axes[0].set_ylabel("Cases")
    axes[1].bar(test["ds"].dt.strftime("%b"), ape, color="#f59e0b")
    axes[1].axhline(ape.mean(), color="#dc2626", ls="--", lw=1.3, label=f"Mean APE={ape.mean():.1f}%")
    axes[1].set_title("Absolute Percentage Error", fontweight="bold"); axes[1].set_ylabel("APE (%)")
    axes[1].legend()
    fig.tight_layout(); fig.savefig(f"{OUT_FIGS}/fig4_residuals.png"); plt.close(fig)

    # Fig 5: components
    fig = result["model"].plot_components(fc_train, figsize=(10, 3.2 * (1 + len(REGRESSORS) > 0)))
    fig.suptitle("Prophet Model Components (log1p-case scale)", y=1.02, fontweight="bold")
    fig.tight_layout(); fig.savefig(f"{OUT_FIGS}/fig5_components.png", bbox_inches="tight"); plt.close(fig)

    # Fig 6: regressor coefficients
    coefs = result["coefs"].sort_values("coef", key=abs)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    colors = np.where(coefs["coef"] >= 0, "#16a34a", "#dc2626")
    ax.barh(coefs["regressor"], coefs["coef"], color=colors)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title("Standardized Regressor Coefficients", fontweight="bold")
    ax.set_xlabel("Coefficient (effect on log1p-cases per 1 SD)")
    fig.tight_layout(); fig.savefig(f"{OUT_FIGS}/fig6_regressor_coefficients.png"); plt.close(fig)

    print(f"\nFigures written to {OUT_FIGS}")


# MAIN
if __name__ == "__main__":
    df = load_data()
    print(f"Loaded {len(df)} months ({df.ds.min().date()} to {df.ds.max().date()})")
    print(f"Regressors used: {REGRESSORS}  (USE_CLIMATE_REGRESSORS={USE_CLIMATE_REGRESSORS})")

    best_cps, best_range, tuning_df = tune_hyperparameters(df)
    result = run_final_holdout(df, best_cps, best_range)
    make_figures(df, result)

    print("\nDone. Tables ->", OUT_TABLES, "| Figures ->", OUT_FIGS)

Loaded 202 months (2009-01-01 to 2025-10-01)
Regressors used: ['y_lag12', 'y_roll12']  (USE_CLIMATE_REGRESSORS=False)
Nested CV: 7 walk-forward folds (train-only, cutoffs 2017-2023)
  cps=0.05 range=0.8  mean_CV_RMSE_log=2.102 +/- 1.066
  cps=0.05 range=0.9  mean_CV_RMSE_log=2.115 +/- 1.090
  cps=0.05 range=1.0  mean_CV_RMSE_log=2.104 +/- 1.081
  cps= 0.1 range=0.8  mean_CV_RMSE_log=2.182 +/- 1.110
  cps= 0.1 range=0.9  mean_CV_RMSE_log=2.171 +/- 1.095
  cps= 0.1 range=1.0  mean_CV_RMSE_log=2.193 +/- 1.102
  cps= 0.3 range=0.8  mean_CV_RMSE_log=2.501 +/- 1.251
  cps= 0.3 range=0.9  mean_CV_RMSE_log=2.607 +/- 1.575
  cps= 0.3 range=1.0  mean_CV_RMSE_log=2.630 +/- 1.565
  cps= 0.5 range=0.8  mean_CV_RMSE_log=2.653 +/- 1.501
  cps= 0.5 range=0.9  mean_CV_RMSE_log=3.176 +/- 1.828
  cps= 0.5 range=1.0  mean_CV_RMSE_log=3.025 +/- 1.826
  cps= 0.7 range=0.8  mean_CV_RMSE_log=2.806 +/- 1.621
  cps= 0.7 range=0.9  mean_CV_RMSE_log=3.472 +/- 1.964
  cps= 0.7 range=1.0  mean_CV_RMSE_log=3.377 +/-